# Analisi certificazioni RIAA
Progetto di Data Science e Laboratorio<br>
*Francesco Cuttini*

In [ ]:
import math
import pylab
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

from IPython.display import HTML

import branca.colormap as cm

snspalette = sns.color_palette(palette='ocean')

sns.set(style="whitegrid", palette=snspalette, font_scale=1.2)

## Il programma di certificazione RIAA
Il "RIAA Gold & Platinum Program" è un'iniziativa della Recording Industry Association of America, costruita intorno alla concessione di premi agli artisti e allo staff di produzione in base alle vendite della loro musica sul territorio degli Stati Uniti d'America.

Ai fini dell'emissione dei premi, la RIAA riconosce tre livelli di certificazione:
 - **Oro**: 500000 unità vendute;
 - **Platino**: 1000000 unità vendute;
 - **Diamante**: 10000000 unità vendute.

Dal 2000 la RIAA mantiene anche il "RIAA Premios de Oro y Platino Program", simile ma dedicato a contenuto almeno al 51% in lingua spagnola:
 - **Oro**: 30000 unità vendute;
 - **Platino**: 60000 unità vendute;
 - **Diamante**: 600000 unità vendute.

## Digitale e *streaming*

A causa della crescente popolarità dell'acquisto musicale (e.g. iTunes), nel 2013 la RIAA ha aggiunto un'apposita scala per le *release* digitali, definita dalle stesse quantità per la certificazione standard.

Un cambiamento importante è stato riconosciuto a causa della crescente popolarità dello *streaming* musicale, e del graduale spostamento dell'industria verso di esso.

Un'unità di certificazione in questo caso equivaleva inizialmente a 100 *stream*.<br>
Questa figura è stata poi alzata a 150 *stream* da febbraio 2026.

Gli altri due programmi, chiamati informalmente *"standard"* e *"latin"*, hanno ricevuto delle modifiche simili nel 2016.
In particolare ai fini delle certificazioni *"standard"*, si considera un'unità di certificazione una dei seguenti:
 1. una vendita permanente di un album digitale o fisico nella sua interezza;
 2. il download di 10 tracce dell'album;
 3. 1500 stream audio o video on-demand di tracce incluse nell'album.

## Domande
Questa presentazione quindi punta a rispondere a qualche domanda che mi sono posto riguardo al programma:

1. Quanto importante è la presenza di artisti non statunitensi premiati per le vendite negli Stati Uniti? Da quali paesi provengono?

2. Com'è cambiato negli anni il panorama dei generi della musica più ascoltata negli Stati Uniti?

3. Com'è cambiata negli anni la quantità di certificazioni emesse?
   - Apertura di Napster e LimeWire
   - Apertura dell'iTunes Store
   - Lancio di Spotify

4. Come ha influito il conteggio degli *streaming* ai fini delle certificazioni sul tempo tra rilascio e prima certificazione RIAA di un'opera?

## Dataset utilizzato
I dati utilizzati per questa analisi provengono da:
 - scraping del sito web RIAA al 13 agosto 2026 ([riaa.com](https://www.riaa.com/gold-platinum/))
   - lista certificazioni
   - date di rilascio (dove disponibili)
 - MusicBrainz API ([musicbrainz.org](https://musicbrainz.org/doc/MusicBrainz_API))
   - artisti e relativi dati
   - generi (dove disponibili)
 - Natural Earth Vector Dataset ([github.com/nvkelso/natural-earth-vector](https://github.com/nvkelso/natural-earth-vector))
   - forme e posizioni delle Nazioni per la visualizzazione in questa presentazione

## Quali sono i paesi con gli artisti più premiati?
*(escludendo gli Stati Uniti d'America per le categorie Standard e Digital)*

In [ ]:
mbareas = pd.read_csv('./outputs/mbareas.csv', encoding='utf-8')
mbareas

jurl = "https://github.com/nvkelso/natural-earth-vector/raw/refs/heads/master/geojson/ne_10m_admin_0_countries.geojson"
countries = gpd.read_file(jurl)
clist = countries[["ISO_A2_EH", "ADMIN", "geometry"]]
countriesgeo = pd.merge(mbareas, clist, left_on='COUNTRY', right_on='ISO_A2_EH', how='right')
countriesgeo = countriesgeo
for k in countriesgeo.keys():
    if k not in ["COUNTRY", "ISO_A2_EH", "ADMIN", "geometry", "LA_SCORE"]:
        countriesgeo.loc[countriesgeo['COUNTRY'] == 'US', k] = 0

In [ ]:
m = folium.Map(location=[40, 0], zoom_start=2, height=500, tiles=None)

css = """
<style>
.leaflet-container {
    background: #ffffff !important;
}

.leaflet-control-zoom {
    display: none !important;
}
</style>

<script>
const run = (e = null) => {
    let legends = document.querySelector(".legend.leaflet-control")
    let checks = document.querySelectorAll(".leaflet-control-layers-list input[type=checkbox]")
    for (let i = 0; i < checks.length; i++) {
        if (e != null && checks[i] != e.target && checks[i].checked && !e.ctrlKey) {
            checks[i].click()
        }
        if (checks[i].checked == true) {
            legends.children[i].style.display = ''
        } else {
            legends.children[i].style.display = 'none'
        }
    }
}

const handler = e => {
    if (e.target.classList.contains("leaflet-control-layers-selector")) {
        if (e.isTrusted) {
            run(e)
        }
    }
}

document.addEventListener("click", handler)

// make sure this runs after the map is initialized (which thanks to folium we don't really know when that is so we kinda have to guess)
setTimeout(run, 500)
setTimeout(run, 750)
setTimeout(run, 1000)
</script>
"""

m.get_root().header.add_child(folium.Element(css))

### it appears folium HATES logarithmic scales so this is me crying
# uses linear interpolation
def find_color_at_percentage(col1, col2, t):
    r = round(col1[0] + (col2[0] - col1[0]) * t)
    g = round(col1[1] + (col2[1] - col1[1]) * t)
    b = round(col1[2] + (col2[2] - col1[2]) * t)
    return (r, g, b)

# kind of a rough guess on how logarithmic scales work. should be enough for our use case.
def create_colors_array(tstart, tend, steps):
    start = (round(255 * tstart[0]), round(255 * tstart[1]), round(255 * tstart[2]))
    end = (round(255 * tend[0]), round(255 * tend[1]), round(255 * tend[2]))
    colors = []
    for i in range(steps):
        perc = i / (steps - 1)
        md = find_color_at_percentage(start, end, perc)
        colors.append(md)
    return colors

# NOTE we assume this is an "rgb" as a tuple of 0-255 values, not what seaborn uses that is 0-1.
#      this is really meant to be used on results of create_colors_array
def colors_rgb_to_hex(colors):
    out = []
    for color in colors:
        # https://stackoverflow.com/a/3380739
        out.append('#%02x%02x%02x' % color)
    return out

def generate_colormap_by_key(key, caption):
    sc = countriesgeo.dropna(subset=[key])
    scz = sc[sc[key] > 0]
    minval = scz[key].min()
    maxval = scz[key].max()
    values = np.geomspace(minval, maxval, num=256).tolist()
    values = [round(i) for i in values]
    legendval = np.geomspace(minval, maxval, num=8).tolist()
    legendval = [round(i) for i in legendval]
    del legendval[1:4] # the scale gets too crowded in the beginning
    colors = colors_rgb_to_hex(create_colors_array((1, 1, 1), snspalette[0], len(values)))
    colormap = cm.StepColormap(colors=colors, index=values, vmin=legendval[0], vmax=legendval[-1])
    colormap.tick_labels = legendval
    colormap.caption = caption
    return colormap

layerst = folium.FeatureGroup(name='Standard')
layerst.add_to(m)
colormapst = generate_colormap_by_key("ST_SCORE", "Numero di dischi di platino nella categoria Standard")
colormapst.add_to(m)

layerdi = folium.FeatureGroup(name='Digital', show=False)
layerdi.add_to(m)
colormapdi = generate_colormap_by_key("DI_SCORE", "Numero di dischi di platino nella categoria Digital")
colormapdi.add_to(m)

layerla = folium.FeatureGroup(name='Latin', show=False)
layerla.add_to(m)
colormapla = generate_colormap_by_key("LA_SCORE", "Numero di dischi di platino nella categoria Latin")
colormapla.add_to(m)

folium.LayerControl(position="topright").add_to(m)

bindings = {
    "ST_SCORE": layerst,
    "DI_SCORE": layerdi,
    "LA_SCORE": layerla
}

# https://geopandas.org/en/stable/gallery/polygon_plotting_with_folium.html
for _, country in countriesgeo.iterrows():
    for key, layer in bindings.items():
        sim_geo = gpd.GeoSeries(country["geometry"]).simplify(tolerance=0.001)
        geo_j = sim_geo.to_json()
        if math.isnan(country[key]) or country[key] <= 0:
            value = '#FFFFFF'
        else:
            value = colormapst(country[key])
        geo_j = folium.GeoJson(data=geo_j, style_function=lambda feature, value=value: {
            "fillColor": value,
            "color": "black",
            "weight": 1,
            "fillOpacity": 1,
        })
        folium.Popup(f"<div style='min-width: 150px;'><h3>{country["ADMIN"]}</h3><p>{int(country[key]) if not math.isnan(country[key]) else 0} dischi di platino totali</p></div>").add_to(geo_j)
        geo_j.add_to(layer)

m.get_root().height = "500px"
m

### Album standard

In [ ]:
key = "ST_SCORE"
clisttbl = countriesgeo[["ADMIN", key]]
clisttbl = clisttbl.sort_values(key, ascending=False)
clisttbl = clisttbl.dropna()
clisttbl = clisttbl.drop_duplicates("ADMIN")
clisttbl[key] = clisttbl[key].astype(int)
clisttbl = clisttbl.rename(columns={ 'ADMIN': 'Paese', f"{key}": 'Dischi di platino' })

HTML(clisttbl[:15].to_html(index=False))

### Album digitali

In [ ]:
key = "DI_SCORE"
clisttbl = countriesgeo[["ADMIN", key]]
clisttbl = clisttbl.sort_values(key, ascending=False)
clisttbl = clisttbl.dropna()
clisttbl = clisttbl.drop_duplicates("ADMIN")
clisttbl[key] = clisttbl[key].astype(int)
clisttbl = clisttbl.rename(columns={ 'ADMIN': 'Paese', f"{key}": 'Dischi di platino' })

HTML(clisttbl[:15].to_html(index=False))

### Album in spagnolo

In [ ]:
key = "LA_SCORE"
clisttbl = countriesgeo[["ADMIN", key]]
clisttbl = clisttbl.sort_values(key, ascending=False)
clisttbl = clisttbl.dropna()
clisttbl = clisttbl.drop_duplicates("ADMIN")
clisttbl[key] = clisttbl[key].astype(int)
clisttbl = clisttbl.rename(columns={ 'ADMIN': 'Paese', f"{key}": 'Dischi di platino' })

HTML(clisttbl[:15].to_html(index=False))

## Andamento per genere

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 5)

genre_trend = pd.read_csv('./outputs/genrtrnd.csv', encoding='utf-8')

genre_totals = genre_trend.sum(axis=0, numeric_only=True)
genre_totals = genre_totals.drop('YEAR')
genre_totals = genre_totals.sort_values(ascending=False)
genre_totals = genre_totals[:10]
genre_totals

p = sns.barplot(data = genre_totals)
p.set(ylabel="certificazioni")

plt.show()

*(numero di certificazioni per i 10 generi più certificati in tutta la cronologia di certificazioni RIAA dove i dati sono ben formati o hanno una corrispondenza in MusicBrainz)*

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 5)

keys = genre_totals.keys()

genre_trend = pd.read_csv('./outputs/genrtrnd.csv', encoding='utf-8')
genre_trend = genre_trend.sort_values("YEAR")
with sns.color_palette("tab10"):
    p = genre_trend.set_index('YEAR')[keys].plot.area()

p.set(xlabel="anni", ylabel="certificazioni emesse")

plt.xlim(1958, 2025)
plt.show()

*(andamento generale dei 10 generi più certificati in tutta la cronologia di certificazioni RIAA)*

### Colonne sonore

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 5)

genre_trend = pd.read_csv('./outputs/sndtrack.csv', encoding='utf-8')

p = sns.lineplot(data=genre_trend, x="YEAR", y="RELEASES")
p.set(xlabel="anni", ylabel="album certificati pubblicati")

plt.show()

*(numero di colonne sonore che hanno ottenuto almeno una certificazione RIAA, per anno di pubblicazione)*

## Certificazioni emesse negli anni

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 5)

relestat = pd.read_csv('./outputs/certstat.csv', encoding='utf-8')
relestat = relestat[relestat['YEAR'] >= 1958][relestat['YEAR'] <= 2025]

p = sns.lineplot(data=relestat, x="YEAR", y="TOTAL")
p.set(xlabel="anni", ylabel="certificazioni")
plt.xlim(1958, 2025)
plt.show()

*(certificazioni RIAA emesse per anno, dati tra gli anni 1958 e 2025)*

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 6)

p = sns.lineplot(data=relestat, x="YEAR", y="TOTAL")
p.set(xlabel="anni", ylabel="certificazioni")

year = 1999
yearval = relestat[relestat['YEAR'] == year]['TOTAL'].values[0]

plt.annotate(
    "Apertura di Napster",
    xy=(year, yearval),
    xytext=(year, yearval - 500),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

year = 2000
yearval = relestat[relestat['YEAR'] == year]['TOTAL'].values[0]

plt.annotate(
    "Apertura di LimeWire",
    xy=(year, yearval),
    xytext=(year, yearval + 500),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

plt.xlim(1985, 2010)
plt.show()

*(dati tra gli anni 1985 e 2010, marcata l'apertura di Napster e LimeWire)*

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 6)

p = sns.lineplot(data=relestat, x="YEAR", y="TOTAL")
p.set(xlabel="anni", ylabel="certificazioni")

year = 2003
yearval = relestat[relestat['YEAR'] == year]['TOTAL'].values[0]

plt.annotate(
    "Rilascio dell'iTunes Store",
    xy=(year, yearval),
    xytext=(year, yearval - 500),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

year = 2004
yearval = relestat[relestat['YEAR'] == year]['TOTAL'].values[0]

plt.annotate(
    "Inizio del tracciamento RIAA delle vendite digitali",
    xy=(year, yearval),
    xytext=(year - 5, yearval + 1000),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

plt.xlim(1985, 2010)
plt.show()

*(dati tra gli anni 1985 e 2010, marcati il rilascio dell'iTunes Store e l'inizio del tracciamento delle vendite digitali)*

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 6)

relestat = pd.read_csv('./outputs/certstat.csv', encoding='utf-8')
relestat = relestat[relestat['YEAR'] >= 1958][relestat['YEAR'] <= 2025]

p = sns.lineplot(data=relestat, x="YEAR", y="TOTAL")
p.set(xlabel="anni", ylabel="certificazioni")

year = 2008
yearval = relestat[relestat['YEAR'] == year]['TOTAL'].values[0]

plt.annotate(
    "Lancio di Spotify",
    xy=(year, yearval),
    xytext=(year, yearval + 600),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

year = 2013
yearval = relestat[relestat['YEAR'] == year]['TOTAL'].values[0]

plt.annotate(
    "Inizio del tracciamento RIAA degli stream",
    xy=(year, yearval),
    xytext=(year, yearval - 500),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

plt.xlim(1995, 2025)

plt.show()

*(dati tra gli anni 1995 e 2025, marcato l'avvio dei conteggi degli streaming ai fini delle certificazioni RIAA)*

## Influenza del conteggio degli *streaming* sui tempi di certificazione

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 5)

relestat = pd.read_csv('./outputs/relestat.csv', encoding='utf-8')
relestat = relestat[relestat['YEAR'] >= 1958]

p = sns.lineplot(data=relestat, x="YEAR", y="DAYS")
p.set(xlabel="anni", ylabel="giorni tra rilascio e prima certificazione")
plt.xlim(1958, 2025)
plt.show()

*(delta mediano tra rilascio e certificazione di un elemento multimediale, dati tra gli anni 1958 e 2025)*

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 6)

trelestat = relestat[relestat['YEAR'] >= 1990]

p = sns.lineplot(data=trelestat, x="YEAR", y="DAYS")
p.set(xlabel="anni", ylabel="giorni tra rilascio e prima certificazione")
plt.xlim(1990, 2025)
plt.show()

*(dati tra gli anni 1990 e 2025)*

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 6)

trelestat = relestat[relestat['YEAR'] >= 1985]
trelestat = trelestat[trelestat['YEAR'] <= 2010]

p = sns.lineplot(data=trelestat, x="YEAR", y="DAYS")
p.set(xlabel="anni", ylabel="giorni tra rilascio e prima certificazione")

year = 1999
yearval = trelestat[trelestat['YEAR'] == year]['DAYS'].values[0]

plt.annotate(
    "Apertura di Napster",
    xy=(year, yearval),
    xytext=(year - 5, yearval + 300),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

year = 2000
yearval = trelestat[trelestat['YEAR'] == year]['DAYS'].values[0]

plt.annotate(
    "Apertura di LimeWire",
    xy=(year, yearval),
    xytext=(year, yearval + 400),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

plt.xlim(1985, 2010)
plt.show()

*(dati tra gli anni 1985 e 2010, marcata l'apertura di Napster e LimeWire)*

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 6)

trelestat = relestat[relestat['YEAR'] >= 1985]
trelestat = trelestat[trelestat['YEAR'] <= 2010]

p = sns.lineplot(data=trelestat, x="YEAR", y="DAYS")
p.set(xlabel="anni", ylabel="giorni tra rilascio e prima certificazione")

year = 2003
yearval = relestat[relestat['YEAR'] == year]['DAYS'].values[0]

plt.annotate(
    "Rilascio dell'iTunes Store",
    xy=(year, yearval),
    xytext=(year, yearval + 100),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

year = 2004
yearval = relestat[relestat['YEAR'] == year]['DAYS'].values[0]

plt.annotate(
    "Inizio tracciamento vendite digitali",
    xy=(year, yearval),
    xytext=(year - 9, yearval + 50),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

plt.xlim(1985, 2010)
plt.show()

*(dati tra gli anni 1985 e 2010, marcati il rilascio dell'iTunes Store e l'inizio del tracciamento delle vendite digitali ai fini delle certificazioni RIAA)*

In [ ]:
%matplotlib inline
pylab.rcParams['figure.figsize'] = (12, 6)

trelestat = relestat[relestat['YEAR'] >= 2000]

p = sns.lineplot(data=trelestat, x="YEAR", y="DAYS")
p.set(xlabel="anni", ylabel="giorni tra rilascio e prima certificazione")

year = 2008
yearval = relestat[relestat['YEAR'] == year]['DAYS'].values[0]

plt.annotate(
    "Lancio di Spotify",
    xy=(year, yearval),
    xytext=(year + 5, yearval - 250),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

year = 2013
yearval = relestat[relestat['YEAR'] == year]['DAYS'].values[0]

plt.annotate(
    "RIAA: Avvio conteggio stream",
    xy=(year, yearval),
    xytext=(year - 6, yearval),
    ha='center',
    bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1),
    arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color="black", lw=1.5)
)

plt.xlim(2000, 2025)

plt.show()

*(dati a partire dal 2000, marcati il lancio di Spotify e l'avvio del conteggio streaming ai fini delle certificazioni RIAA)*

## Conclusioni

 - C'è una presenza marcata di pubblicazioni estere all'interno del registro di certificazioni RIAA, in particolare da paesi anglofoni.
 - La cronologia delle certificazioni è stata accompagnata da una presenza annuale particolarmente massiccia di musica rock, poi andata a ridursi sempre più per lasciare spazio ad altri generi.
   - Le colonne sonore rilasciate nell'ultimo decennio, probabilmente anche agevolate dal conteggio degli *streaming*, raggiungono più spesso le soglie di certificazione RIAA.
 - Il conteggio degli *streaming*, come ipotizzato, ha agevolato il raggiungimento delle soglie di certificazione da parte degli album.
   - È stato osservato che, prima dell'avvio del conteggio degli *streaming* ma dopo l'apertura di Spotify, il rilascio di certificazioni ha subito un forte calo, correlato ad un probabile rallentamento delle vendite permanenti fisiche o digitali.